# PICKO Research · NB3 V2 — **Semantic separation**: can it *learn* to route by domain without the source name?

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab first: Runtime → Change runtime type → GPU (L4 recommended; T4/A100 also fine).**
This cell clones **upstream Needle** (Cactus, pinned commit) and installs it, clones this **PICKO** repo,
pins the exact JAX/Flax, mounts Drive, and points **both** the data (in) and the checkpoints+results (out)
at your **`MyDrive/picko/`** folder — so a runtime restart loses nothing.

**Prerequisite (one-time):** `picko_training_pool.jsonl` must be in `MyDrive/picko/`. **Running locally?**
This cell is a no-op — install Needle yourself (`pip install -e /path/to/needle`) and skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
# Upstream Needle (Cactus) — de-vendored: cloned + installed at a pinned commit.
NEEDLE_REPO = "https://github.com/cactus-compute/needle.git"
NEEDLE_SHA  = "34861f39ae292429f80a62c96abe83218a852d57"   # pinned; has _per_tool_split — update if upstream drifts
# This PICKO repo — the scripts/notebooks/data imported below.
PICKO_REPO   = "https://github.com/HadarBit/picko.git"     # <- set to your submission repo URL
PICKO_BRANCH = "main"
if IN_COLAB:
    if not os.path.exists("/content/needle"):
        !git clone -q {NEEDLE_REPO} /content/needle && cd /content/needle && git checkout -q {NEEDLE_SHA}
    if not os.path.exists("/content/picko"):
        !git clone -q -b {PICKO_BRANCH} {PICKO_REPO} /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    %pip install -q -e /content/needle                          # install upstream Needle
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    import shutil
    DRIVE = "/content/drive/MyDrive/picko"                      # <- everything lives here
    os.environ["PICKO_OUT_DIR"] = f"{DRIVE}/picko_out"          # checkpoints + results (durable)
    os.environ["PICKO_LOG"]     = f"{DRIVE}/picko_out/run.log"  # durable log across restarts
    os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    dst = "/content/picko/data/picko_training_pool.jsonl"
    if not os.path.exists(dst):
        cands = [f"{DRIVE}/picko_training_pool.jsonl", "/content/drive/MyDrive/picko_training_pool.jsonl"]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            have = os.listdir(DRIVE) if os.path.isdir(DRIVE) else "(MyDrive/picko not found)"
            raise FileNotFoundError(
                "picko_training_pool.jsonl not found. Upload it to MyDrive/picko/. "
                f"Currently in {DRIVE}: {have}")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
        print("copied data from", src)
    try:                                                        # guard: upstream must expose the split PICKO uses
        from needle.training.finetune import _per_tool_split    # noqa: F401
    except Exception as e:
        raise ImportError(f"Upstream Needle @ {NEEDLE_SHA[:7]} lacks _per_tool_split ({e}). "
                          "Pin NEEDLE_SHA to a commit that has it, or re-run the install.")
    import jax
    print("GPU:");
    !nvidia-smi -L
    print("jax devices:", jax.devices())
    _plat = jax.devices()[0].platform
    assert _plat == "gpu", (
        f"JAX is running on '{_plat}', NOT the GPU — every finetune/eval will be ~30x slower "
        "(hours instead of minutes). FIX: Runtime > Change runtime type > GPU (L4), then "
        "Runtime > Restart session, and re-run this cell. If a GPU IS selected but this still "
        "fails, the CUDA plugin didn't load — re-run the %pip lines above, then restart.")
    print(f"bootstrap OK · GPU active · needle@{NEEDLE_SHA[:7]} · data =", dst, "· OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally. Install upstream Needle first: pip install -e /path/to/needle")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()
env_report(OUT_DIR)   # jax devices + is OUT_DIR durable (Drive)?

## 2 · The question

NB3 found look-alike **search** tools easy to tell apart across sources — but most training queries **name
the source** (58-85% of `arxiv/pubmed/wikipedia` search queries contain the source word), so that result may
reflect **lexical matching**, not understanding.

**V2 asks a stronger question: can the model *learn* to route by domain meaning alone?** We take a pair that
does the same action in disjoint domains — `arxiv_search_papers` (computer science) vs
`pubmed_search_articles` (medicine) — with freshly written queries that **never name the source**, only the
topic, and **train a focused 2-tool model** on them.

- **Total train/test separation** — the same deterministic `per_tool_split` holds out 10 queries per tool
  (20 total), so the model can never memorise a test answer.
- **Baseline (before):** the already-trained focus40 model on the same held-out test — how well the
  name-trained model routes source-free queries *without* semantic training.
- **After:** the 2-tool model trained on the source-free difference, on the same test.

The **before→after** gap is the finding: if the baseline is near chance but training on the clean semantic
signal lifts it well above 50%, the distinction **is learnable** and the original data simply never taught
it.

## 3 · Build the source-free dataset

Reads the two domain query files (`data/semantic/cs.json` → arxiv, `data/semantic/medicine.json` → pubmed),
drops any query that still names a source (safety net), and bakes the offered pair (full schema, order
randomised) into each row — the standard `{query, tools, answers}` format. Only the **unnamed** (source-free)
queries are used here. Falls back to the prebuilt `data/semantic/probe.jsonl` if the raw files aren't present.

In [ ]:
import re, collections, random as _rnd
BANNED = r"\b(arxiv|arxiv\.org|pubmed|medline|ncbi|pmid|biorxiv|medrxiv|preprint|wikipedia|wiki|hugging\s*face|huggingface|semantic scholar)\b"
DOMAIN = {"arxiv_search_papers": ("cs", "arXiv"), "pubmed_search_articles": ("medicine", "PubMed")}
RAWFILE = {"arxiv_search_papers": "semantic/cs.json", "pubmed_search_articles": "semantic/medicine.json"}
PAIR_TOOLS = list(DOMAIN)                       # the forced 2-way choice

def _find(name):
    for root in [os.path.join(ROOT, "data"), "/content/drive/MyDrive/picko",
                 "/content/drive/MyDrive/picko/data"]:
        p = os.path.join(root, name)
        if os.path.exists(p): return p
    return None

def _tools_json(seed):                          # offered pair (full schema), order randomised per row
    ts = [cat.by_name[n] for n in PAIR_TOOLS]
    _rnd.Random(seed).shuffle(ts)
    return json.dumps(ts, separators=(",", ":"), ensure_ascii=False)

raw_paths = {g: _find(f) for g, f in RAWFILE.items()}
if all(raw_paths.values()):
    DATA, seen, i = [], set(), 0
    for gold, (domain, src) in DOMAIN.items():
        ans = json.dumps([{"name": gold, "arguments": {}}])
        for q in json.load(open(raw_paths[gold])):
            q = q.strip(); key = re.sub(r"\W+", " ", q.lower()).strip()
            if not q or key in seen or re.search(BANNED, q, re.I): continue
            seen.add(key)
            DATA.append({"query": q, "tools": _tools_json(i), "answers": ans,
                         "gold": gold, "domain": domain}); i += 1
    pp = os.path.join(ROOT, "data", "semantic", "train.jsonl")
    os.makedirs(os.path.dirname(pp), exist_ok=True)
    with open(pp, "w") as f:
        for r in DATA: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    log(f"built source-free dataset ({len(DATA)} rows) → {pp}")
else:
    pp = _find("semantic/probe.jsonl")
    if not pp: raise FileNotFoundError("need data/semantic/cs.json + medicine.json (or probe.jsonl) in data/ or Drive")
    DATA = [r for r in (json.loads(l) for l in open(pp) if l.strip()) if r.get("condition", "unnamed") == "unnamed"]
    log(f"loaded {len(DATA)} source-free rows from {pp}")

df = pd.DataFrame(DATA)
print("dataset:", len(DATA), "| per domain:", df["domain"].value_counts().to_dict())
display(df[["query", "gold", "domain"]].head(6))

## 4 · Configure

In [ ]:
NB_DIR = os.path.join(OUT_DIR, "nb3"); os.makedirs(NB_DIR, exist_ok=True)   # this notebook's outputs
EPOCHS        = 3      # small 2-tool set -> a few passes
BATCH_SIZE    = 8
MAX_GEN_LEN   = 64     # we only score tool SELECTION
RUN_TRAIN     = True
FORCE_RETRAIN = False
BASELINE_CKPT = next((c for c in [os.path.join(OUT_DIR, "nb2", "picko_depth_focus40_best.pkl"),
                                  os.path.join(OUT_DIR, "nb1", "picko_breadth_focus40_best.pkl")]
                      if os.path.exists(c)), None)
print("pair:", PAIR_TOOLS, "| epochs:", EPOCHS, "| baseline:", BASELINE_CKPT, "| out:", NB_DIR)

## 5 · Train / test split (total separation)

The same deterministic `per_tool_split` (`seed=42`) both notebook and training subprocess use: 10 test + 10
val per tool, the rest train. The test 20 are never seen in training.

In [ ]:
train, val, TEST = per_tool_split(DATA)
print(f"train={len(train)}  val={len(val)}  test={len(TEST)}")
print("test per domain:", collections.Counter(e["domain"] for e in TEST))

## 6 · Baseline — the existing model on the held-out test (before semantic training)

In [ ]:
import contextlib, io
if BASELINE_CKPT:
    bm, bp, btk = load_model(BASELINE_CKPT)
    with contextlib.redirect_stdout(io.StringIO()):
        base_preds = predict(bm, bp, btk, TEST, max_gen_len=MAX_GEN_LEN, batch=BATCH_SIZE)
    base_m = evaluate(TEST, base_preds, family_of=family_of)
    log(f"BASELINE (name-trained model, no semantic training): selection={base_m['selection_acc']:.3f}")
else:
    base_preds, base_m = None, None
    log("no baseline checkpoint found (run nb1/nb2 first) — skipping the 'before' comparison")

## 7 · Train the 2-tool semantic model\nTrained on the source-free difference only; the returned held-out `test` + `preds` are scored below. Resumable to Drive.

In [ ]:
R = finetune_and_eval(cat, raw, tok, PAIR_TOOLS, "semantic_pair", NB_DIR,
                      dataset=DATA, epochs=EPOCHS, run_train=RUN_TRAIN,
                      force_retrain=FORCE_RETRAIN, max_gen_len=MAX_GEN_LEN, batch_size=BATCH_SIZE)
after_test, after_preds, after_m = R["test"], R["preds"], R["metrics"]
log(f"AFTER (trained on the semantic difference): selection={after_m['selection_acc']:.3f}")

## 8 · Before vs after, per domain

In [ ]:
def _by_domain(examples, preds):
    sc = evaluate_per_example(examples, preds)
    d = pd.DataFrame({"domain": [e["domain"] for e in examples], "selected": [s["selected"] for s in sc]})
    return d.groupby("domain")["selected"].mean().round(4).to_dict()

rowsout = []
if base_m:
    rowsout.append({"phase": "before (baseline)", "overall": round(base_m["selection_acc"], 4), **_by_domain(TEST, base_preds)})
rowsout.append({"phase": "after (semantic)", "overall": round(after_m["selection_acc"], 4), **_by_domain(after_test, after_preds)})
summary = pd.DataFrame(rowsout)
RES = os.path.join(NB_DIR, "semantic_learn_results.json")
json.dump({"pair": PAIR_TOOLS, "summary": rowsout, "confusion_after": confusion(after_test, after_preds)},
          open(RES, "w"), indent=2)
log(f"saved {RES}")
display(summary)

## 9 · Plot

In [ ]:
cats = ["overall", "cs", "medicine"]; x = np.arange(len(cats)); w = 0.8 / max(len(rowsout), 1)
palette = {"before (baseline)": "#8a8a8a", "after (semantic)": "#009E73"}
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
for j, r in enumerate(rowsout):
    vals = [r.get(c, np.nan) for c in cats]
    off = (j - (len(rowsout) - 1) / 2) * w
    ax.bar(x + off, vals, w, color=palette.get(r["phase"], "#0072B2"), edgecolor="white", lw=0.6, label=r["phase"])
    for k, v in enumerate(vals):
        if not np.isnan(v): ax.text(x[k] + off, min(v + 0.03, 1.03), f"{v:.2f}", ha="center", fontsize=9)
ax.axhline(0.5, color="#C44E52", ls="--", lw=1.2, label="chance (2-way)")
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylim(0, 1.08); ax.set_ylabel("Tool-selection accuracy")
ax.set_title("Can PICKO Learn to Route by Domain Without the Source Name?"); ax.legend(loc="lower right")
if sns: sns.despine(ax=ax)
ax.grid(axis="y", color="#cccccc", lw=0.6, alpha=0.6); ax.set_axisbelow(True)

conf = confusion(after_test, after_preds)
M = pd.DataFrame(0, index=PAIR_TOOLS, columns=PAIR_TOOLS)
for rr, row in conf.items():
    for p, c in row.items():
        if rr in PAIR_TOOLS and p in PAIR_TOOLS: M.loc[rr, p] = c
short = lambda n: n.replace("_search_papers", "").replace("_search_articles", "")
if sns:
    sns.heatmap(M.div(M.sum(1).replace(0, 1), axis=0), cmap="Greens", vmin=0, vmax=1, cbar=False,
                annot=M.values, fmt="d", linewidths=0.5, linecolor="white", ax=ax2,
                xticklabels=[short(c) for c in M.columns], yticklabels=[short(r) for r in M.index])
ax2.set_title("After training: routing"); ax2.set_xlabel("predicted"); ax2.set_ylabel("true domain tool")
plt.tight_layout(); save_fig("semantic_learnability", out_dir=NB_DIR); plt.show()

## 10 · Read-out

Offered only the two same-action tools on source-free queries, routing can come only from the domain
meaning. The **baseline** shows how the name-trained model copes without the source word; the **after** bar
shows the focused 2-tool model trained on the clean semantic difference, on a **held-out** test it never
saw. If after-training accuracy sits well above the 50% chance line — and above the baseline — PICKO **can**
learn domain-based routing; if it stays near chance, the distinction is beyond what this signal teaches the
26M model. The confusion panel shows whether residual errors are symmetric or collapse toward one domain.